## tl;dr

v0.2.0 → v0.3.0 的两语向 COMET 变化均小于 0.12 分，整体水平接近。新版原生导出与 AAR 译文一致。完整 app 三轮已取得 24 个进程、72 遍输出，但四个耗时指标波动超限，尚未通过稳定性验收。本笔记本逐项复核归档，不连接手机或重新下载模型。

## Context & Methods

### Key Assumptions

- FLORES-200 devtest 前 200 条，固定模型、中文参考与 `wmt22-comet-da × 100`。
- 旧/新各自默认路径，每方向各三个原生进程；范围不是置信区间，也不是统计等价检验。
- app 与 native 的速度、PSS/RSS 分开，失败记录不拼接成完整场次。
- `verify.py` 为本笔记本的只读检查实现，可独立运行；Python 标准库即可，展示依赖 IPython。
- 来源与序列号脱敏映射见 README；所有被评分译文逐文件验 SHA-256。

## Data

### 1. Load the local verification helpers

In [1]:
from pathlib import Path
import json
import runpy
from IPython.display import Markdown, display

root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / 'tools/app-bench/run.py').is_file())
record = root / 'benchmarks/v0.3.0/mi14-2026-09-09/reconnected'
checks = runpy.run_path(str(record / 'verify.py'))


### 2. Verify complete exports, hashes, score lineage and reference identity

In [2]:
quality_rows = checks['quality_checks']()
print('Verified: 12/12 native exports; all 42 scored calls and 8 unique translation sets; model, corpus and source-file hashes.')

Verified: 12/12 native exports; all 42 scored calls and 8 unique translation sets; model, corpus and source-file hashes.


## Results

### 3. Rebuild the old/new quality table

In [3]:
rows = ['| Direction | v0.2.0 range | v0.3.0 | New − old range |', '|---|---:|---:|---:|']
for item in quality_rows:
    lo, hi = item['old_comet_range']
    dlow, dhigh = item['new_minus_old_range']
    rows.append(f"| {item['direction']} | {lo:.2f}–{hi:.2f} | {item['new_comet']:.2f} | {dlow:+.2f} to {dhigh:+.2f} |")
display(Markdown('\n'.join(rows)))

| Direction | v0.2.0 range | v0.3.0 | New − old range |
|---|---:|---:|---:|
| enzh | 87.15–87.19 | 87.27 | +0.08 to +0.12 |
| jazh | 86.78–86.82 | 86.71 | -0.10 to -0.06 |

### 4. Inspect changed translations without conflating them with quality loss

In [4]:
for item in quality_rows:
    print(item['direction'], 'old repeated-run changed rows:', item['old_repeats_changed_rows'],
          'old → new changed rows:', item['old_to_new_changed_rows'])
print('New native output is identical across all three processes and matches the previously scored AAR outputs.')

enzh old repeated-run changed rows: [0, 5, 5] old → new changed rows: [37, 41, 41]
jazh old repeated-run changed rows: [0, 14, 14] old → new changed rows: [38, 36, 36]
New native output is identical across all three processes and matches the previously scored AAR outputs.


### 5. Re-run performance evidence checks

In [5]:
race = checks['app_checks'](record / 'app-read-race')
assert race['accepted'] is False
print('Read/write-race attempt: rejected as incomplete.')
current = checks['current_app_checks']()
verdict = current['verdict']
assert current['scored_output_matched_passes'] == 72
assert len(verdict['errors']) == 4 and verdict['accepted'] is False
assert all(s['successful_processes'] == 3 for s in verdict['scenarios'])
print('All 72 fresh passes match the previously scored outputs:', current['comet_x100'])
print('Current full-session acceptance:', verdict['accepted'])
print('Issues:', verdict['errors'])

Read/write-race attempt: rejected as incomplete.
All 72 fresh passes match the previously scored outputs: {'enzh/mlkit': 72.69342935085297, 'enzh/bergamot': 87.26716721057892, 'jazh/mlkit': 68.92763012647629, 'jazh/bergamot': 86.71308651566505}
Current full-session acceptance: False
Issues: ['enzh/bergamot/1t: cold_ms spread exceeds 10%', 'enzh/bergamot/4t: cold_ms spread exceeds 10%', 'jazh/bergamot/1t: warm_ms spread exceeds 10%', 'jazh/bergamot/4t: cold_ms spread exceeds 10%']


### 6. Rebuild all eight performance rows (reference only, not accepted baseline)

In [6]:
rows = ['| Scenario | First inputs/s | Warm inputs/s | First PSS MiB | Warm PSS MiB |', '|---|---:|---:|---:|---:|']
for s in verdict['scenarios']:
    rows.append(f"| {s['scenario']} | {s['cold_inputs_per_second']:.2f} | {s['warm_inputs_per_second']:.2f} | {s['cold_pss_mib']:.1f} | {s['warm_pss_mib']:.1f} |")
display(Markdown('\n'.join(rows)))
for direction in ['enzh', 'jazh']:
    ml = next(s for s in verdict['scenarios'] if s['scenario'] == f'{direction}/mlkit/Nonet')
    bg = next(s for s in verdict['scenarios'] if s['scenario'] == f'{direction}/bergamot/1t')
    print(direction, 'first speed ratio:', round(ml['cold_ms']/bg['cold_ms'], 4),
          'first PSS ratio:', round(bg['cold_pss_mib']/ml['cold_pss_mib'], 4))

| Scenario | First inputs/s | Warm inputs/s | First PSS MiB | Warm PSS MiB |
|---|---:|---:|---:|---:|
| enzh/mlkit/Nonet | 12.86 | 13.02 | 187.1 | 192.2 |
| enzh/bergamot/1t | 66.55 | 66.10 | 264.1 | 233.2 |
| enzh/bergamot/2t | 106.38 | 121.68 | 378.3 | 321.3 |
| enzh/bergamot/4t | 178.68 | 230.71 | 601.2 | 601.9 |
| jazh/mlkit/Nonet | 6.14 | 6.16 | 239.1 | 239.6 |
| jazh/bergamot/1t | 29.11 | 35.33 | 359.2 | 337.0 |
| jazh/bergamot/2t | 50.07 | 61.33 | 551.3 | 506.1 |
| jazh/bergamot/4t | 85.94 | 107.81 | 933.9 | 934.8 |

enzh first speed ratio: 5.1731 first PSS ratio: 1.4115
jazh first speed ratio: 4.739 first PSS ratio: 1.5023


### 7. Keep native acceptance separate from app measurements

In [7]:
if (record / 'native-results.json').exists():
    print(checks['native_checks']())
else:
    print('Native result not archived yet; no acceptance claim.')

{'exit_code': 2, 'accepted': False, 'diagnostic': 'INVALID: v0.2.0: missing or unexpected scenarios'}


## Takeaways

本批语料支持“整体 COMET 水平接近”，不支持“每条译文完全不变”；日→中的小幅分数下降也保留在表中。新版与 ML Kit 的质量优势只针对端侧 SDK 和本批语料。性能表必须结合完整性、温度与样本波动检查，不能由翻译质量结果代替验收。